# 02_data_exploration

This notebook validates loaded datasets and performs initial exploratory analysis. It assumes the ingestion notebook has already loaded or created the raw CSV files in `data/raw/`.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
RAW_DIR = Path("data/raw")

def load_csv_dataset(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"File not found: {path}")
        return None

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as exc:
        print(f"Failed to read {path.name}: {exc}")
        return None

    for col in df.columns:
        if df[col].dtype == object:
            converted = pd.to_datetime(df[col], errors="ignore", infer_datetime_format=True, dayfirst=True)
            if pd.api.types.is_datetime64_any_dtype(converted):
                df[col] = converted
                print(f"Converted {path.name}.{col} to datetime64[ns]")
    return df

def summarize_dataframe(df: pd.DataFrame, name: str):
    print(f"
=== Summary for {name} ===")
    print(f"Shape: {df.shape}")
    print(df.dtypes)
    print("Missing values per column:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())

def inspect_date_columns(df: pd.DataFrame, name: str):
    date_cols = [col for col in df.columns if pd.api.types.is_datetime64_any_dtype(df[col])]
    if not date_cols:
        print(f"No parsed datetime columns found in {name}.")
        return
    print(f"Parsed datetime columns in {name}: {date_cols}")
    for col in date_cols:
        print(f"  {col}: min={df[col].min()}, max={df[col].max()}
,
,
        print("Could not validate AMFI codes because scheme_code is missing in one of the datasets.")
        return
    master_codes = set(master_df["scheme_code"].dropna().astype(str).unique())
    nav_codes = set(nav_history_df["scheme_code"].dropna().astype(str).unique())
    missing_in_nav = sorted(master_codes - nav_codes)
    missing_in_master = sorted(nav_codes - master_codes)
    print("
=== AMFI code validation ===")
    print(f"Missing from nav_history: {len(missing_in_nav)}")
    if missing_in_nav:
        print(missing_in_nav[:50])
    print(f"Missing from fund_master: {len(missing_in_master)}")
    if missing_in_master:
        print(missing_in_master[:50])

def explore_fund_master(df: pd.DataFrame):
    print("
=== Fund master exploration ===")
    for column in ["fund_house", "category", "sub_category", "risk_grade"]:
        if column in df.columns:
            unique_values = df[column].dropna().unique()
            print(f"{column}: {len(unique_values)} unique values")
            print(sorted(unique_values)[:20])
        else:
            print(f"Column '{column}' not found in fund_master.")

In [ ]:
fund_master_path = RAW_DIR / "01_fund_master.csv"
nav_history_path = RAW_DIR / "02_nav_history.csv"

fund_master = load_csv_dataset(fund_master_path)
nav_history = load_csv_dataset(nav_history_path)

if fund_master is not None:
    summarize_dataframe(fund_master, "fund_master")
    inspect_date_columns(fund_master, "fund_master")
    explore_fund_master(fund_master)

if nav_history is not None:
    summarize_dataframe(nav_history, "nav_history")
    inspect_date_columns(nav_history, "nav_history")

if fund_master is not None and nav_history is not None:
    validate_amfi_codes(fund_master, nav_history)